# Evaluation of three waypoint occupation-matching algorithms

This notebook runs and compares:

1. `waypoint_block_to_block_discrete.py`
2. `waypoint_block_to_history.py`
3. `waypoint_block_to_history_dynamic_astar.py`

The primary comparison is the **trajectory objective versus elapsed physical time**. Additional diagnostics cover objective versus block index, travel and patrol allocation, contraction behavior, route lengths, time-to-threshold, and final occupation residuals.

> The scripts use different output names. The loader below normalizes them into one common result structure.

## 1. Imports and experiment configuration

Set `RUN_SCRIPTS = True` to launch fresh runs. For a quick smoke test, start with 100 to 500 blocks. The full defaults of 30,000 to 40,000 blocks can be expensive on a 128 × 128 grid.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path.cwd()
RESULTS_ROOT = BASE / "comparison_runs"
RESULTS_ROOT.mkdir(exist_ok=True)

RUN_SCRIPTS = False
BLOCKS = 500
SEED = 13
MAX_HORIZON = 500
VERBOSE_EVERY = 100
SPEED = 0.35
SELF_LOOP_TIME = 0.01
MAX_BLOCK_TIME = 0.0  # 0 disables the physical-time cap in history scripts

METHODS = {
    "Discrete block-to-block": {
        "script": BASE / "waypoint_block_to_block_discrete.py",
        "output": RESULTS_ROOT / "discrete_block_to_block",
        "extra_args": [],
    },
    "Block-to-history": {
        "script": BASE / "waypoint_block_to_history.py",
        "output": RESULTS_ROOT / "block_to_history",
        "extra_args": ["--speed", str(SPEED), "--self-loop-time", str(SELF_LOOP_TIME),
                       "--max-block-time", str(MAX_BLOCK_TIME)],
    },
    "Dynamic A* block-to-history": {
        "script": BASE / "waypoint_block_to_history_dynamic_astar.py",
        "output": RESULTS_ROOT / "dynamic_astar_block_to_history",
        "extra_args": ["--speed", str(SPEED), "--self-loop-time", str(SELF_LOOP_TIME),
                       "--max-block-time", str(MAX_BLOCK_TIME)],
    },
}

for name, cfg in METHODS.items():
    if not cfg["script"].exists():
        raise FileNotFoundError(f"Missing script for {name}: {cfg['script']}")

METHODS

## 2. Run the scripts

This cell uses the same seed, block count, and horizon for all three methods. Each method writes to its own output directory.

In [ ]:
def run_method(name, cfg):
    cfg["output"].mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, str(cfg["script"]),
        "--blocks", str(BLOCKS),
        "--max-horizon", str(MAX_HORIZON),
        "--seed", str(SEED),
        "--verbose-every", str(VERBOSE_EVERY),
        "--output-dir", str(cfg["output"]),
        *cfg["extra_args"],
    ]
    print()
    print(f"=== {name} ===")
    print(" ".join(cmd))
    started = time.perf_counter()
    completed = subprocess.run(cmd, text=True, capture_output=True)
    elapsed = time.perf_counter() - started
    print(completed.stdout[-3000:])
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"{name} failed with exit code {completed.returncode}")
    return elapsed

wall_times = {}
if RUN_SCRIPTS:
    for name, cfg in METHODS.items():
        wall_times[name] = run_method(name, cfg)
    pd.Series(wall_times, name="wall_time_seconds").to_frame()
else:
    print("RUN_SCRIPTS is False. Set it to True and rerun this cell to generate results.")

## 3. Normalize output files

For the discrete method, the physically meaningful curve is `trajectory_objective_history.npy` against `physical_time_history.npy`. For both block-to-history methods, the corresponding files are `objective_history.npy` and `time_history.npy`.

In [ ]:
def first_existing(directory, names):
    for name in names:
        path = directory / name
        if path.exists():
            return path
    return None

def load_json_if_present(path):
    return json.loads(path.read_text()) if path.exists() else {}

def load_method(name, cfg):
    out = cfg["output"]
    time_path = first_existing(out, ["physical_time_history.npy", "time_history.npy"])
    objective_path = first_existing(out, ["trajectory_objective_history.npy", "objective_history.npy"])
    if time_path is None or objective_path is None:
        raise FileNotFoundError(
            f"Missing result arrays for {name} in {out}. Run the scripts first."
        )
    physical_time = np.load(time_path)
    trajectory_objective = np.load(objective_path)
    if len(physical_time) != len(trajectory_objective):
        raise ValueError(f"Length mismatch for {name}: time={len(physical_time)}, objective={len(trajectory_objective)}")

    discrete_objective_path = out / "objective_history.npy"
    discrete_objective = None
    if name == "Discrete block-to-block" and discrete_objective_path.exists():
        discrete_objective = np.load(discrete_objective_path)

    csv_path = out / "block_history.csv"
    records = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()
    target = np.load(out / "target.npy") if (out / "target.npy").exists() else None
    final_eta = np.load(out / "final_trajectory_empirical_measure.npy") if (out / "final_trajectory_empirical_measure.npy").exists() else None
    if final_eta is None and (out / "final_empirical_measure.npy").exists():
        final_eta = np.load(out / "final_empirical_measure.npy")
    summary = load_json_if_present(out / "summary.json")

    return {
        "name": name,
        "output": out,
        "physical_time": physical_time,
        "trajectory_objective": trajectory_objective,
        "discrete_objective": discrete_objective,
        "records": records,
        "target": target,
        "final_eta": final_eta,
        "summary": summary,
    }

results = {name: load_method(name, cfg) for name, cfg in METHODS.items()}
{name: (len(r["physical_time"]), r["physical_time"][-1], r["trajectory_objective"][-1])
 for name, r in results.items()}

## 4. Summary metrics

In [ ]:
def time_to_threshold(t, g, threshold):
    hit = np.flatnonzero(g <= threshold)
    return float(t[hit[0]]) if len(hit) else np.nan

thresholds = [1e-1, 1e-2, 1e-3, 1e-4]
rows = []
for name, r in results.items():
    t = r["physical_time"]
    g = r["trajectory_objective"]
    rec = r["records"]
    row = {
        "method": name,
        "blocks": len(g) - 1,
        "final physical time": t[-1],
        "final trajectory objective": g[-1],
        "minimum trajectory objective": np.min(g),
        "time at minimum": t[np.argmin(g)],
        "objective increases": int(np.sum(np.diff(g) > 0)),
    }
    for threshold in thresholds:
        row[f"time to G <= {threshold:g}"] = time_to_threshold(t, g, threshold)
    if not rec.empty:
        contracted_col = "contracted" if "contracted" in rec else None
        row["contraction rate"] = rec[contracted_col].mean() if contracted_col else np.nan
        travel_col = "physical_travel_time" if "physical_travel_time" in rec else "travel_time"
        patrol_col = "physical_patrol_time" if "physical_patrol_time" in rec else "patrol_time"
        row["travel-time fraction"] = rec[travel_col].sum() / max(rec[travel_col].sum() + rec[patrol_col].sum(), 1e-15)
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index("method")
summary_df

## 5. Main comparison: trajectory objective versus physical time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
for name, r in results.items():
    t = r["physical_time"]
    g = r["trajectory_objective"]
    axes[0].plot(t, g, label=name)
    positive = (t > 0) & (g > 0)
    axes[1].loglog(t[positive], g[positive], label=name)

axes[0].set(xlabel="Physical time", ylabel=r"Trajectory objective $G(\eta_T)$",
            title="Objective versus physical time")
axes[1].set(xlabel="Physical time", ylabel=r"Trajectory objective $G(\eta_T)$",
            title="Objective versus physical time, log-log")
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 6. Objective versus feedback/block iteration

This separates optimization progress per gradient update from progress per unit of physical time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
for name, r in results.items():
    g = r["trajectory_objective"]
    k = np.arange(len(g))
    axes[0].plot(k, g, label=name)
    positive = g > 0
    axes[1].semilogy(k[positive], g[positive], label=name)
axes[0].set(xlabel="Block / feedback iteration", ylabel=r"$G(\eta)$", title="Objective per feedback update")
axes[1].set(xlabel="Block / feedback iteration", ylabel=r"$G(\eta)$", title="Objective per feedback update, semilog-y")
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Discrete optimizer objective versus executed trajectory objective

This diagnostic is available for the fully discrete controller and directly measures the execution gap.

In [ ]:
r = results["Discrete block-to-block"]
if r["discrete_objective"] is not None:
    t = r["physical_time"]
    gd = r["discrete_objective"]
    gp = r["trajectory_objective"]
    m = min(len(t), len(gd), len(gp))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
    axes[0].loglog(t[:m], gd[:m], label="Discrete objective")
    axes[0].loglog(t[:m], gp[:m], label="Executed trajectory objective")
    axes[0].set(xlabel="Physical time", ylabel=r"$G$", title="Discrete plan versus physical execution")
    gap = gp[:m] - gd[:m]
    axes[1].plot(t[:m], gap)
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set(xlabel="Physical time", ylabel="Trajectory G - discrete G", title="Execution gap")
    for ax in axes:
        ax.grid(True, which="both", alpha=0.3)
        ax.legend() if ax is axes[0] else None
    plt.tight_layout()
    plt.show()
else:
    print("No discrete objective history found.")

## 8. Travel, patrol, and block-time diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for name, r in results.items():
    df = r["records"]
    if df.empty:
        continue
    travel_col = "physical_travel_time" if "physical_travel_time" in df else "travel_time"
    patrol_col = "physical_patrol_time" if "physical_patrol_time" in df else "patrol_time"
    block_col = "physical_block_time" if "physical_block_time" in df else "block_time"
    axes[0].plot(df.index, df[travel_col], alpha=0.75, label=name)
    axes[1].plot(df.index, df[patrol_col], alpha=0.75, label=name)
    axes[2].plot(df.index, df[block_col], alpha=0.75, label=name)
axes[0].set(title="Travel time per block", xlabel="Block", ylabel="Time")
axes[1].set(title="Patrol / hold time per block", xlabel="Block", ylabel="Time")
axes[2].set(title="Total physical block time", xlabel="Block", ylabel="Time")
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 9. Route length and contraction-ratio diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.0))
for name, r in results.items():
    df = r["records"]
    if df.empty:
        continue
    route_col = "travel_length" if "travel_length" in df else "route_cells"
    ratio_col = "contraction_ratio" if "contraction_ratio" in df else "ratio"
    axes[0].plot(df.index, df[route_col], alpha=0.75, label=name)
    finite = np.isfinite(df[ratio_col].to_numpy(dtype=float))
    axes[1].plot(df.index[finite], df.loc[finite, ratio_col], alpha=0.75, label=name)
axes[0].set(title="Route length", xlabel="Block", ylabel="Cells / transitions")
axes[1].set(title="Frozen-gradient contraction ratio", xlabel="Block", ylabel="Ratio")
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 10. Objective change per block and per unit physical time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.0))
for name, r in results.items():
    t = r["physical_time"]
    g = r["trajectory_objective"]
    dt = np.diff(t)
    dg = np.diff(g)
    axes[0].plot(np.arange(len(dg)), dg, alpha=0.7, label=name)
    valid = dt > 0
    axes[1].plot(np.arange(len(dg))[valid], (-dg[valid] / dt[valid]), alpha=0.7, label=name)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(title="Objective change per block", xlabel="Block", ylabel=r"$\Delta G$")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(title="Objective reduction rate", xlabel="Block", ylabel=r"$-\Delta G / \Delta T$")
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 11. Final occupation distributions and residuals

In [ ]:
available = [(name, r) for name, r in results.items() if r["target"] is not None and r["final_eta"] is not None]
if available:
    nrows = len(available)
    fig, axes = plt.subplots(nrows, 3, figsize=(14, 4.2 * nrows), squeeze=False)
    for row, (name, r) in enumerate(available):
        target = r["target"]
        eta = r["final_eta"]
        side = int(round(np.sqrt(len(target))))
        target_img = target.reshape(side, side)
        eta_img = eta.reshape(side, side)
        residual = eta_img - target_img
        lim = max(abs(residual.min()), abs(residual.max()), 1e-15)
        im0 = axes[row, 0].imshow(target_img, origin="lower", cmap="viridis")
        im1 = axes[row, 1].imshow(eta_img, origin="lower", cmap="viridis")
        im2 = axes[row, 2].imshow(residual, origin="lower", cmap="coolwarm", vmin=-lim, vmax=lim)
        axes[row, 0].set_title(f"{name}: target")
        axes[row, 1].set_title(f"{name}: final physical occupation")
        axes[row, 2].set_title(f"{name}: residual")
        fig.colorbar(im0, ax=axes[row, 0], fraction=0.046)
        fig.colorbar(im1, ax=axes[row, 1], fraction=0.046)
        fig.colorbar(im2, ax=axes[row, 2], fraction=0.046)
    plt.tight_layout()
    plt.show()

## 12. Fair comparison on a common physical-time interval

Methods may stop at different physical times. This cell truncates all curves to their shared time horizon and interpolates them onto one logarithmic grid.

In [ ]:
positive_starts = [r["physical_time"][r["physical_time"] > 0][0] for r in results.values()]
common_start = max(positive_starts)
common_end = min(r["physical_time"][-1] for r in results.values())

if common_end > common_start:
    common_time = np.geomspace(common_start, common_end, 500)
    common_curves = {}
    for name, r in results.items():
        common_curves[name] = np.interp(common_time, r["physical_time"], r["trajectory_objective"])

    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    for name, curve in common_curves.items():
        ax.loglog(common_time, curve, label=name)
    ax.set(xlabel="Common physical time", ylabel=r"$G(\eta_T)$",
           title="Comparison on shared physical-time support")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

    common_final = pd.Series({name: curve[-1] for name, curve in common_curves.items()},
                             name=f"G at common T={common_end:.6g}")
    display(common_final.sort_values().to_frame())
else:
    print("The methods do not have overlapping positive physical-time support.")

## 13. Export comparison tables

The notebook writes normalized summary metrics and per-method histories to `comparison_runs/`.

In [ ]:
summary_path = RESULTS_ROOT / "comparison_summary.csv"
summary_df.to_csv(summary_path)

for name, r in results.items():
    safe = name.lower().replace(" ", "_").replace("*", "astar").replace("-", "_")
    history = pd.DataFrame({
        "physical_time": r["physical_time"],
        "trajectory_objective": r["trajectory_objective"],
    })
    history.to_csv(RESULTS_ROOT / f"{safe}_normalized_history.csv", index=False)

print(f"Saved: {summary_path.resolve()}")